## 1. Mount Google Drive and Setup Project Directories

In [ ]:
from google.colab import drive
import os

print("--- 💾 Step 1: Mount Google Drive ---")

try:
    # Use force_remount=True to resolve potential connection issues
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"⚠️ Warning: {e}")
    print("If the drive is already mounted, you can safely ignore this error.")

# Define and create the main directory for your dataset (Update this path as needed)
output_dir = "/content/drive/MyDrive/path/to/your/dataset"
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Output directory is ready: {output_dir}")

## 2. Install Required Python Libraries

In [ ]:
# Install necessary geospatial and data science libraries with the quiet flag
!pip install sentinelhub -q
!pip install rasterio -q
!pip install pyproj -q
!pip install pandas -q
!pip install openpyxl -q

print("✅ Libraries installed successfully.")

## 3. Configure Copernicus Data Space Ecosystem Authentication

In [ ]:
from sentinelhub import SHConfig
import getpass

# 1. Initialize an empty configuration object
config = SHConfig()

# 2. Securely prompt the user for credentials
config.sh_client_id = getpass.getpass("Enter your Copernicus Data Space Client ID: ")
config.sh_client_secret = getpass.getpass("Enter your Copernicus Data Space Client Secret: ")

# 3. Configure endpoints for the Copernicus Data Space service
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"

# 4. Save configuration for the current session
config.save()

print("✅ Authentication configured successfully and credentials saved for the current session.")

## 4. Extract Central Coordinates for All Regions from the Benchmark Dataset
 Reference dataset: Earth Science Informatics (2025) - Sentinel-2 and Sentinel-3 Spatiotemporal Fusion Benchmark

In [ ]:
import os
import pandas as pd
import rasterio
from sentinelhub import BBox, CRS
from tqdm.notebook import tqdm
from pyproj import Transformer, CRS as ProjCRS

print("--- 🗺️ Step 4: Extracting Central Coordinates for All Regions ---")

# 1. Define the source path for the benchmark dataset (Update this path according to your local structure)
# Based on: Earth Science Informatics (2025), DOI: 10.1007/s12145-025-01855-4
source_dataset_path = "/content/drive/MyDrive/path/to/benchmark_dataset"

if not os.path.exists(source_dataset_path):
    print(f"❌ Error: Path '{source_dataset_path}' not found.")
    raise SystemExit("Source path not found.")

unique_regions = {}  # Dictionary to store unique regions

print(f"Scanning {source_dataset_path} to identify regions...")

# 2. Scan directories to find reference raster files and extract their center coordinates
for root, dirs, files in tqdm(os.walk(source_dataset_path), desc="Scanning Regions"):
    rel_path = os.path.relpath(root, source_dataset_path)
    path_parts = rel_path.split(os.sep)

    if len(path_parts) >= 1 and path_parts[0] != '.':
        region_name = path_parts[0]

        if region_name in unique_regions:
            continue

        s2_ref_file = None
        for f in files:
            if "_s2_10m.tif" in f:
                s2_ref_file = os.path.join(root, f)
                break

        if s2_ref_file:
            try:
                with rasterio.open(s2_ref_file) as src:
                    bounds_utm = src.bounds
                    crs_utm = src.crs

                    center_x = (bounds_utm.left + bounds_utm.right) / 2
                    center_y = (bounds_utm.top + bounds_utm.bottom) / 2

                    transformer_to_wgs84 = Transformer.from_crs(crs_utm, ProjCRS("EPSG:4326"), always_xy=True)
                    center_lon, center_lat = transformer_to_wgs84.transform(center_x, center_y)

                    unique_regions[region_name] = {
                        'Region': region_name,
                        'Center_Lon': center_lon,
                        'Center_Lat': center_lat
                    }
            except Exception as e:
                print(f"⚠️ Error reading file {s2_ref_file}: {e}")

# --- 3. Add Custom / Additional Regions ---
new_regions = [
    {"Region": "france_paris", "Center_Lon": 2.3522, "Center_Lat": 48.8566},
    {"Region": "spain_madrid", "Center_Lon": -3.7038, "Center_Lat": 40.4168},
    {"Region": "germany_berlin", "Center_Lon": 13.4050, "Center_Lat": 52.5200},
    {"Region": "egypt_cairo", "Center_Lon": 31.2357, "Center_Lat": 30.0444}
]
print(f"✅ Added {len(new_regions)} custom regions.")

for region in new_regions:
    if region['Region'] not in unique_regions:
        unique_regions[region['Region']] = region

# --- 4. Define Final Parameters ---
df_regions_coords = pd.DataFrame(list(unique_regions.values()))

# Define BBox dimensions (100 km)
BBOX_SIZE_METERS = 100000

if not df_regions_coords.empty:
    print(f"\n✅ Successfully identified {len(df_regions_coords)} unique regions:")
    print(df_regions_coords.head())
    print(f"\n✅ BBox Size: {BBOX_SIZE_METERS} x {BBOX_SIZE_METERS} meters.")
    print("\nStep 4 completed. Variables 'df_regions_coords' and 'BBOX_SIZE_METERS' are ready.")
else:
    print("❌ No regions found.")

## 5. Multi-Year Bulk Search and Scene Matching (Sentinel-2 & Sentinel-3)

In [ ]:
import pandas as pd
from sentinelhub import SentinelHubCatalog, DataCollection, BBox, CRS, Geometry
from datetime import datetime, timedelta, timezone
from dateutil.parser import parse
from tqdm.notebook import tqdm
import os
from pyproj import Transformer, CRS as ProjCRS

print("--- 🛰️ Step 5: Multi-Year Bulk Search and Matching ---")

# --- Step 1: Verify Required Variables ---
try:
    _ = config             # From Step 3
    _ = df_regions_coords  # From Step 4
    _ = BBOX_SIZE_METERS   # From Step 4
except NameError as e:
    print(f"❌ Error: Required variable not found: {e}")
    raise RuntimeError("Please execute the previous setup steps first.")

# --- Step 2: Define Search Parameters ---
START_DATE = "2018-01-01"
END_DATE = "2025-01-01"
CLOUD_COVER_THRESHOLD = 5.0
S3_SEARCH_WINDOW_HOURS = 2

# Update this output path as needed for your repository/environment
final_results_csv_path = "/content/drive/MyDrive/path/to/search_results_FINAL.csv"
print(f"Search time span: {START_DATE} to {END_DATE}")
print(f"Results will be saved to: {final_results_csv_path}")

# --- Step 3: Define Helper Functions ---
def get_collection(base_enum, name, service_url="https://sh.dataspace.copernicus.eu"):
    safe_name = f"{name}_multiyear_final"
    if hasattr(DataCollection, safe_name): return getattr(DataCollection, safe_name)
    try: return base_enum.define_from(name=safe_name, service_url=service_url)
    except ValueError:
        for m in DataCollection:
            if m.value.service_url == service_url and m.value.api_id == base_enum.value.api_id: return m
        raise

def search_catalog(catalog, collection, bbox, time_interval, query=None):
    return list(catalog.search(
        collection, bbox=bbox, time=time_interval, filter=query,
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"]}
    ))

# --- Step 4: Initialize Catalog and Collections ---
catalog = SentinelHubCatalog(config=config)
s2_l1c_col = get_collection(DataCollection.SENTINEL2_L1C, "S2_L1C")
s2_l2a_col = get_collection(DataCollection.SENTINEL2_L2A, "S2_L2A")
s3_olci_col = get_collection(DataCollection.SENTINEL3_OLCI, "S3_OLCI")
s3_slstr_col = get_collection(DataCollection.SENTINEL3_SLSTR, "S3_SLSTR")

# --- Step 5: Resume Logic Implementation ---
all_valid_pairs_df = pd.DataFrame()
processed_regions = []

if os.path.exists(final_results_csv_path):
    try:
        all_valid_pairs_df = pd.read_csv(final_results_csv_path)
        processed_regions = all_valid_pairs_df['Region'].unique().tolist()
        print(f"\n✅ Previous progress loaded. {len(processed_regions)} regions already processed.")
        print(f"   Skipped regions: {processed_regions}")
    except Exception as e:
        print(f"⚠️ Warning: Previous results file was corrupted and ignored. {e}")
        all_valid_pairs_df = pd.DataFrame()
        processed_regions = []
else:
    print("\nℹ️ No previous results file found. Starting search from scratch.")

# --- Step 6: Main Processing Loop (with Resume Support) ---
print(f"\n--- 🚀 Starting Multi-Year Search for {len(df_regions_coords)} Regions ---")

for index, row in tqdm(df_regions_coords.iterrows(), total=len(df_regions_coords), desc="Processing Regions"):

    region_name = row['Region']
    center_lon = row['Center_Lon']
    center_lat = row['Center_Lat']

    # Skip already processed regions
    if region_name in processed_regions:
        print(f"\n📍 Region: {region_name} (Already processed, skipping)")
        continue

    print(f"\n📍 Region: {region_name} (Processing...)")
    new_pairs_for_this_region = []

    try:
        # 1. Construct custom BBox
        utm_crs_sh = CRS.get_utm_from_wgs84(center_lon, center_lat)
        crs_utm = ProjCRS(utm_crs_sh.epsg)
        transformer_to_utm = Transformer.from_crs(ProjCRS("EPSG:4326"), crs_utm, always_xy=True)

        center_x, center_y = transformer_to_utm.transform(center_lon, center_lat)
        half_size_meters = BBOX_SIZE_METERS / 2
        utm_bbox_coords = [
            center_x - half_size_meters, center_y - half_size_meters,
            center_x + half_size_meters, center_y + half_size_meters
        ]
        bbox_utm = BBox(utm_bbox_coords, crs=utm_crs_sh)
        bbox_wgs84 = bbox_utm.transform(CRS.WGS84)

        # 2. Search for Sentinel-2 scenes within the multi-year range
        s2_query = f"eo:cloud_cover <= {CLOUD_COVER_THRESHOLD}"

        s2_results = search_catalog(
            catalog, s2_l2a_col, bbox_wgs84, (START_DATE, END_DATE), s2_query
        )
        print(f"   - S2 scenes found (Cloud < {CLOUD_COVER_THRESHOLD}%): {len(s2_results)}")

        if not s2_results:
            continue

        # 3. Check for matching Sentinel-3 pairs for each S2 scene
        for s2_item in s2_results:
            s2_dt = parse(s2_item['properties']['datetime']).astimezone(timezone.utc)
            s2_cloud = s2_item['properties']['eo:cloud_cover']

            s3_interval = (
                (s2_dt - timedelta(hours=S3_SEARCH_WINDOW_HOURS)).isoformat(),
                (s2_dt + timedelta(hours=S3_SEARCH_WINDOW_HOURS)).isoformat()
            )

            olci_hits = list(catalog.search(s3_olci_col, bbox=bbox_wgs84, time=s3_interval, limit=1))
            if not olci_hits: continue
            slstr_hits = list(catalog.search(s3_slstr_col, bbox=bbox_wgs84, time=s3_interval, limit=1))
            if not slstr_hits: continue

            olci_dt = parse(olci_hits[0]['properties']['datetime'])
            s2_dl_interval = ((s2_dt - timedelta(seconds=30)).isoformat(), (s2_dt + timedelta(seconds=30)).isoformat())
            olci_dl_interval = ((olci_dt - timedelta(seconds=30)).isoformat(), (olci_dt + timedelta(seconds=30)).isoformat())
            slstr_dl_interval = ((parse(slstr_hits[0]['properties']['datetime']) - timedelta(seconds=30)).isoformat(), (parse(slstr_hits[0]['properties']['datetime']) + timedelta(seconds=30)).isoformat())

            new_pairs_for_this_region.append({
                'Region': region_name,
                'Date_Str': s2_dt.strftime('%Y%m%d'),
                'S2_DateTime': s2_dt,
                'S2_Cloud': s2_cloud,
                'BBox_UTM_Bounds': list(bbox_utm.geometry.bounds),
                'CRS_UTM_String': utm_crs_sh.ogc_string(),
                'S2_L1C_Time_Start': s2_dl_interval[0], 'S2_L1C_Time_End': s2_dl_interval[1],
                'S2_L2A_Time_Start': s2_dl_interval[0], 'S2_L2A_Time_End': s2_dl_interval[1],
                'S3_OLCI_Time_Start': olci_dl_interval[0], 'S3_OLCI_Time_End': olci_dl_interval[1],
                'S3_SLSTR_Time_Start': slstr_dl_interval[0], 'S3_SLSTR_Time_End': slstr_dl_interval[1],
                'OLCI_Found_Time': olci_dt
            })

        # Save progress incrementally
        if new_pairs_for_this_region:
            df_new = pd.DataFrame(new_pairs_for_this_region)
            df_new.to_csv(final_results_csv_path, mode='a', header=not os.path.exists(final_results_csv_path), index=False)
            all_valid_pairs_df = pd.concat([all_valid_pairs_df, df_new])
            print(f"   ✅ Found and appended {len(new_pairs_for_this_region)} valid S2/S3 pairs for this region.")
        else:
            print("   ℹ️ No matching S3 pairs found for the retrieved S2 scenes.")

    except Exception as e:
        print(f"❌ Error processing region {region_name}: {e}")
        continue

# --- Step 7: Final Summary ---
print("\n==================================================")
print(f"🏁🏁🏁 Multi-Year Bulk Search Completed.")
if not all_valid_pairs_df.empty:
    print(f"✅ Total valid pairs found across all regions: {len(all_valid_pairs_df)}")
    print(f"📁 Final CSV results list saved to:")
    print(f"   {final_results_csv_path}")
else:
    print("❌ No matching image pairs found meeting the specified criteria.")

## 6. Smart Multi-Sensor Data Download and Processing Pipeline (with Resume Capability)

In [ ]:
import numpy as np
import pandas as pd
from sentinelhub import CRS, BBox, SentinelHubRequest, DataCollection, MimeType, bbox_to_dimensions
import rasterio
from rasterio.transform import from_bounds
from dateutil.parser import parse
import os
from tqdm.notebook import tqdm
import ast
from datetime import timedelta

print("--- 🛰️ Step 6: Smart Data Download and Processing Pipeline ---")

# --- Step 1: Load and Prepare Download List ---
results_csv_path = "/content/drive/MyDrive/path/to/search_results_FINAL.csv"

if 'df_multiyear_list_FINAL' in locals() and not df_multiyear_list_FINAL.empty:
    print("✅ Using existing download list from memory.")
    df_to_download = df_multiyear_list_FINAL.copy()
elif os.path.exists(results_csv_path):
    print("✅ Reading complete download list from saved CSV file...")
    df_to_download = pd.read_csv(results_csv_path)

    # Robust parsing of datetime columns with ISO8601 fallback
    try:
        df_to_download['S2_DateTime'] = pd.to_datetime(df_to_download['S2_DateTime'], format='ISO8601')
        df_to_download['OLCI_Found_Time'] = pd.to_datetime(df_to_download['OLCI_Found_Time'], format='ISO8601')
    except ValueError:
        print("   Using 'mixed' format parser for complex timestamp strings...")
        df_to_download['S2_DateTime'] = pd.to_datetime(df_to_download['S2_DateTime'], format='mixed')
        df_to_download['OLCI_Found_Time'] = pd.to_datetime(df_to_download['OLCI_Found_Time'], format='mixed')
else:
    raise RuntimeError("❌ Download list (search_results_FINAL.csv) not found! Please execute Step 5 first.")

# --- Step 2: Define Output Parameters and Constants ---
output_base_path = "/content/drive/MyDrive/path/to/downloaded_dataset"
os.makedirs(output_base_path, exist_ok=True)
s2_patch_resolution = 50
s3_patch_resolution = 500
BAD_SCL_VALUES = [1, 2, 3, 6, 8, 9, 10, 11]

# --- Step 3: Define Evalscripts ---
evalscript_s2_l1c_data = """
    //VERSION=3
    function setup() { return { input: [{ bands: ["B02","B03","B04","B8A","B11","B12"], units: "DN" }], output: { bands: 6, sampleType: "FLOAT32" } }; }
    function toReflectance(dn) { return dn * 0.0001; }
    function evaluatePixel(sample) { return [ toReflectance(sample.B02), toReflectance(sample.B03), toReflectance(sample.B04), toReflectance(sample.B8A), toReflectance(sample.B11), toReflectance(sample.B12) ]; }
"""
evalscript_s2_l2a_mask = """
    //VERSION=3
    function setup() { return { input: [{ bands: ["SCL"] }], output: { bands: 1, sampleType: "UINT8" } }; }
    function evaluatePixel(sample) { return [ sample.SCL ]; }
"""
evalscript_olci_4band = """
    //VERSION=3
    function setup() { return { input: [{ bands: ["B04", "B06", "B08", "B17"] }], output: { bands: 4, sampleType: "FLOAT32" } }; }
    function evaluatePixel(sample) { return [ sample.B04, sample.B06, sample.B08, sample.B17 ]; }
"""
evalscript_slstr_2band = """
    //VERSION=3
    function setup() { return { input: [{ bands: ["S5", "S6"] }], output: { bands: 2, sampleType: "FLOAT32" } }; }
    function evaluatePixel(sample) { return [ sample.S5, sample.S6 ]; }
"""
print("Evalscripts initialized successfully.")

# --- Step 4: Helper Functions and Collection Setup ---
def get_collection(base_enum, name, service_url="https://sh.dataspace.copernicus.eu"):
    if hasattr(DataCollection, name): return getattr(DataCollection, name)
    try: return base_enum.define_from(name=name, service_url=service_url)
    except ValueError:
        for m in DataCollection:
            if m.value.service_url == service_url and m.value.api_id == base_enum.value.api_id: return m
        import random
        return base_enum.define_from(name=f"{name}_{random.randint(0,9999)}", service_url=service_url)

def save_geotiff(filepath, array, bbox, crs_str):
    h, w, c = array.shape
    west, south, east, north = bbox.geometry.bounds
    transform = from_bounds(west, south, east, north, w, h)
    with rasterio.open(filepath, 'w', driver='GTiff', height=h, width=w, count=c, dtype=array.dtype, crs=crs_str, transform=transform, nodata=np.nan) as dst:
        dst.write(array.transpose(2, 0, 1))

s2_l1c_col = get_collection(DataCollection.SENTINEL2_L1C, "S2_L1C_DL")
s2_l2a_col = get_collection(DataCollection.SENTINEL2_L2A, "S2_L2A_DL")
s3_olci_col = get_collection(DataCollection.SENTINEL3_OLCI, "S3_OLCI_DL")
s3_slstr_col = get_collection(DataCollection.SENTINEL3_SLSTR, "S3_SLSTR_DL")
print("Catalogs and collections are ready.")

# --- Step 5: Main Download Loop ---
print(f"\n--- 🚀 Starting download process for {len(df_to_download)} image pairs ---")

for index, row in tqdm(df_to_download.iterrows(), total=len(df_to_download), desc="Downloading"):

    try:
        region = row['Region']
        date_str = str(row['Date_Str'])

        folder_path = os.path.join(output_base_path, region, date_str)
        os.makedirs(folder_path, exist_ok=True)

        s2_dt = pd.to_datetime(row['S2_DateTime'])
        olci_dt = pd.to_datetime(row['OLCI_Found_Time'])
        s2_time_str = s2_dt.strftime("%Y%m%dT%H%M%S")
        s3_time_str = olci_dt.strftime("%Y%m%dT%H%M%S")

        s2_50m_filename = f"S2_6Band_TOA_Masked_{s2_time_str}_50m.tif"
        s3_500m_filename = f"S3_6Band_TOA_{s3_time_str}_500m.tif"

        path_s2_50 = os.path.join(folder_path, s2_50m_filename)
        path_s3_500 = os.path.join(folder_path, s3_500m_filename)

        # Skip if both files already exist (Resume support)
        if os.path.exists(path_s2_50) and os.path.exists(path_s3_500):
            continue

        bounds = row['BBox_UTM_Bounds']
        if isinstance(bounds, str): bounds = ast.literal_eval(bounds)
        crs_str = row['CRS_UTM_String']
        bbox = BBox(bounds, crs=CRS(crs_str))

        s2_l1c_t = (row['S2_L1C_Time_Start'], row['S2_L1C_Time_End'])
        s2_l2a_t = (row['S2_L2A_Time_Start'], row['S2_L2A_Time_End'])
        olci_t = (row['S3_OLCI_Time_Start'], row['S3_OLCI_Time_End'])
        slstr_t = (row['S3_SLSTR_Time_Start'], row['S3_SLSTR_Time_End'])

        # 1. Download and process Sentinel-2 (50m)
        if not os.path.exists(path_s2_50):
            s2_size_50m = bbox_to_dimensions(bbox, resolution=s2_patch_resolution)
            req_data_50 = SentinelHubRequest(
                evalscript=evalscript_s2_l1c_data,
                input_data=[SentinelHubRequest.input_data(data_collection=s2_l1c_col, time_interval=s2_l1c_t)],
                responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                bbox=bbox, size=s2_size_50m, config=config
            )
            data_50m = req_data_50.get_data()[0]

            req_mask_50 = SentinelHubRequest(
                evalscript=evalscript_s2_l2a_mask,
                input_data=[SentinelHubRequest.input_data(data_collection=s2_l2a_col, time_interval=s2_l2a_t)],
                responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                bbox=bbox, size=s2_size_50m, config=config
            )
            mask_50m = req_mask_50.get_data()[0]

            mask_bool_50 = np.isin(mask_50m.squeeze(), BAD_SCL_VALUES)
            mask_3d_50 = np.repeat(mask_bool_50[:, :, np.newaxis], 6, axis=2)
            data_50m[mask_3d_50] = np.nan

            save_geotiff(path_s2_50, data_50m, bbox, crs_str)

        # 2. Download and process Sentinel-3 (500m)
        if not os.path.exists(path_s3_500):
            s3_size = bbox_to_dimensions(bbox, resolution=s3_patch_resolution)
            req_olci = SentinelHubRequest(
                evalscript=evalscript_olci_4band,
                input_data=[SentinelHubRequest.input_data(data_collection=s3_olci_col, time_interval=olci_t)],
                responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                bbox=bbox, size=s3_size, config=config
            )
            data_olci = req_olci.get_data()[0]

            req_slstr = SentinelHubRequest(
                evalscript=evalscript_slstr_2band,
                input_data=[SentinelHubRequest.input_data(data_collection=s3_slstr_col, time_interval=slstr_t)],
                responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                bbox=bbox, size=s3_size, config=config
            )
            data_slstr = req_slstr.get_data()[0]

            data_s3_final = np.dstack((data_olci, data_slstr))

            if np.isnan(np.nanmin(data_s3_final)):
                print(f"   ❌ Error: Sentinel-3 data was empty for {region}/{date_str}. Cleaning up...")
                if os.path.exists(path_s2_50): os.remove(path_s2_50)
                continue

            save_geotiff(path_s3_500, data_s3_final, bbox, crs_str)

    except Exception as e:
        print(f"   ❌ Error processing {region}/{date_str}: {e}")
        continue

print("\n🏁 Download process completed successfully.")

## 7. Quality Check and Visual Inspection of Downloaded Dataset Samples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import os
import glob
import random

print("--- 📊 Dataset Visual Inspection and Quality Check ---")

# --- Step 1: Define Downloaded Dataset Path ---
dataset_path = "/content/drive/MyDrive/path/to/downloaded_dataset"

if not os.path.exists(dataset_path):
    print(f"❌ Error: Directory '{dataset_path}' not found.")
    raise SystemExit("Please execute the download step first.")

# --- Step 2: Discover Folders Containing Valid Images ---
all_folders = []
for root, dirs, files in os.walk(dataset_path):
    if any(f.startswith("S2_") and f.endswith("_50m.tif") for f in files):
        all_folders.append(root)

if not all_folders:
    raise RuntimeError("❌ No image folders found!")

print(f"✅ Found {len(all_folders)} directories containing images.")

# --- Step 3: Select Random Samples for Inspection ---
num_samples = 3
selected_folders = random.sample(all_folders, min(num_samples, len(all_folders)))

# Normalization helper function for visualization
def normalize(array):
    valid = array[~np.isnan(array)]
    if valid.size == 0: return np.zeros_like(array.transpose(1, 2, 0))
    p2, p98 = np.nanpercentile(valid, [2, 98])
    norm = np.clip((array - p2) / (p98 - p2 + 1e-6), 0, 1)
    return norm.transpose(1, 2, 0)

# --- Step 4: Inspect and Visualize Samples ---
for folder in selected_folders:
    try:
        # Locate files
        s2_path = glob.glob(os.path.join(folder, "S2_*_50m.tif"))[0]
        s3_path = glob.glob(os.path.join(folder, "S3_*_500m.tif"))[0]

        folder_name = f"{os.path.basename(os.path.dirname(folder))}/{os.path.basename(folder)}"
        print(f"\n=======================================================")
        print(f"📍 Sample: {folder_name}")

        # Read Sentinel-2 data
        with rasterio.open(s2_path) as src_s2:
            s2_data = src_s2.read()
            h_s2, w_s2 = src_s2.height, src_s2.width
            print(f"  🛰️ Sentinel-2 (50m):")
            print(f"    Dimensions: {h_s2} x {w_s2} pixels")
            print(f"    Min: {np.nanmin(s2_data):.4f} | Max: {np.nanmax(s2_data):.4f}")
            print(f"    Masked Pixels (NaN): {np.isnan(s2_data).sum()} (out of {s2_data.size})")

        # Read Sentinel-3 data
        with rasterio.open(s3_path) as src_s3:
            s3_data = src_s3.read()
            h_s3, w_s3 = src_s3.height, src_s3.width
            print(f"  🛰️ Sentinel-3 (500m):")
            print(f"    Dimensions: {h_s3} x {w_s3} pixels (Expected ~1/10 of Sentinel-2)")
            print(f"    Min: {np.nanmin(s3_data):.4f} | Max: {np.nanmax(s3_data):.4f}")

        # Plotting
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

        # S2 RGB (Bands 3, 2, 1 -> R, G, B mapping)
        ax1.imshow(normalize(s2_data[[2, 1, 0], :, :]))
        ax1.set_title(f"Sentinel-2 (Masked)\n{h_s2}x{w_s2}")
        ax1.axis('off')

        # S3 RGB (Bands 2, 1, 0)
        ax2.imshow(normalize(s3_data[[2, 1, 0], :, :]), interpolation='nearest')
        ax2.set_title(f"Sentinel-3 (Raw)\n{h_s3}x{w_s3}")
        ax2.axis('off')

        plt.show()

    except Exception as e:
        print(f"❌ Error displaying sample from {folder}: {e}")

print("\n✅ Visual inspection completed.")